In [14]:
import matplotlib.pyplot as plt
from matplotlib import style
style.use('ggplot')
import numpy as np
from sklearn.cluster import MeanShift
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import pandas as pd


df = pd.read_excel('../../Data/titanic.xls')
original_df =pd.DataFrame.copy(df)
df.drop(['body','name'], axis=1, inplace=True)
df = df.infer_objects()
df.fillna(0, inplace=True)


def handle_non_numerical_data(df):
    columns = df.columns.values

    for column in columns:
        text_digit_vals = {}
        def convert_to_int(val): 
            return text_digit_vals[val]
        
        if df[column].dtype != np.int64 and df[column].dtype != np.float64:
            column_contents = df[column].values.tolist()
            unique_elements = set(column_contents)
            x=0
            for unique in unique_elements:
                if unique not in text_digit_vals:
                    text_digit_vals[unique] =x
                    x+=1
            
            df[column] = list(map(convert_to_int, df[column]))
    
    return df


df = handle_non_numerical_data(df)
df.head()

,pclass,survived,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,home.dest
0,1,1,1,29.0000,0,0,747,211.3375,79,1,1,250
1,1,1,0,0.9167,1,2,509,151.5500,63,1,8,175
2,1,0,1,2.0000,1,2,509,151.5500,63,1,0,175
3,1,0,0,30.0000,1,2,509,151.5500,63,1,0,175
4,1,0,1,25.0000,1,2,509,151.5500,63,1,0,175


In [15]:
from sklearn.cluster import MeanShift
from sklearn import preprocessing
import numpy as np

# Drop irrelevant columns and prepare data
df.drop(['boat', 'sex'], axis=1, inplace=True)
X = np.array(df.drop(['survived'], axis=1).astype(float))
X = preprocessing.scale(X)
y = np.array(df['survived'])

# Perform clustering
clf = MeanShift()
clf.fit(X)

labels = clf.labels_
cluster_centers = clf.cluster_centers_

# Assign cluster labels to the original DataFrame
original_df['cluster_group'] = np.nan
for i in range(len(X)):
    original_df.loc[i, 'cluster_group'] = labels[i]

# Count the number of unique clusters
n_clusters_ = len(np.unique(labels))

# Calculate survival rates
survival_rates = {}
for i in range(n_clusters_):
    temp_df = original_df[original_df['cluster_group'] == i]
    if len(temp_df) > 0:
        survival_cluster = temp_df[temp_df['survived'] == 1]
        survival_rate = len(survival_cluster) / len(temp_df)
        survival_rates[i] = survival_rate
    else:
        survival_rates[i] = 0

print(survival_rates)


{0: 0.3766025641025641, 1: 1.0, 2: 0.5121951219512195, 3: 0.0, 4: 0.5}


In [16]:
print(original_df[(original_df['cluster_group'] ==1)])

     pclass  survived                                               name  \
49        1         1                 Cardeza, Mr. Thomas Drake Martinez   
50        1         1  Cardeza, Mrs. James Warburton Martinez (Charlo...   
66        1         1                        Chaudanson, Miss. Victorine   
129       1         1                               Geiger, Miss. Amalie   
183       1         1                             Lesurer, Mr. Gustave J   
302       1         1                                   Ward, Miss. Anna   

        sex   age  sibsp  parch    ticket      fare        cabin embarked  \
49     male  36.0      0      1  PC 17755  512.3292  B51 B53 B55        C   
50   female  58.0      0      1  PC 17755  512.3292  B51 B53 B55        C   
66   female  36.0      0      0  PC 17608  262.3750          B61        C   
129  female  35.0      0      0    113503  211.5000         C130        C   
183    male  35.0      0      0  PC 17755  512.3292         B101        C   
302  

In [17]:
print(original_df[(original_df['cluster_group'] ==4)])

     pclass  survived                                               name  \
115       1         0                                  Fortune, Mr. Mark   
116       1         1                Fortune, Mrs. Mark (Mary McDougald)   
252       1         0                         Ryerson, Mr. Arthur Larned   
253       1         1    Ryerson, Mrs. Arthur Larned (Emily Maria Borie)   
644       3         0         Asplund, Mr. Carl Oscar Vilhelm Gustafsson   
646       3         1  Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...   

        sex   age  sibsp  parch    ticket      fare            cabin embarked  \
115    male  64.0      1      4     19950  263.0000      C23 C25 C27        S   
116  female  60.0      1      4     19950  263.0000      C23 C25 C27        S   
252    male  61.0      1      3  PC 17608  262.3750  B57 B59 B63 B66        C   
253  female  48.0      1      3  PC 17608  262.3750  B57 B59 B63 B66        C   
644    male  40.0      1      5    347077   31.3875           